# Simple
Find the simplest possible way to serialize this data into RDF

In [ ]:
from lxml import etree

# Load the XML file
tree = etree.parse('demarc_graph/data/elliottDiss.xml')
root = tree.getroot()
print(root.tag)  # Print the root element tag

{urn:oasis:names:tc:opendocument:xmlns:office:1.0}document-content


In [14]:

xml_namespaces = {'text': 'urn:oasis:names:tc:opendocument:xmlns:text:1.0'}
instance_headings = root.findall(".//text:h[@text:style-name='treInstance']", xml_namespaces)

from rdflib import Graph, URIRef, Literal, Namespace
from rdflib.namespace import RDF as NS_RDF, RDFS as NS_RDFS, XSD as NS_XSD, DCTERMS as NS_DCTERMS, FOAF as NS_FOAF


g = Graph()

# about the author
NS_PAREGORIOS = Namespace("https://paregorios.org/")
TOM = NS_PAREGORIOS["me"]
NS_ORCID = Namespace("https://orcid.org/")
TOM_ORCID = NS_ORCID["0000-0002-4114-6677"]
g.add((TOM, NS_RDF.type, NS_FOAF.Person))
g.add((TOM, NS_FOAF.firstName, Literal("Tom")))
g.add((TOM, NS_FOAF.surname, Literal("Elliott")))
g.add((TOM, NS_FOAF.isPrimaryTopicOf, TOM_ORCID))
# hcommons? email? VIAF?
g.add((TOM, NS_FOAF.workInfoHomepage, URIRef("https://isaw.nyu.edu/people/staff/tom-elliott")))

# about the work
demarc_uri = "https://paregorios.org/demarc/db"
DEMARC = URIRef(demarc_uri)

#g.add((URIRef(demarc_uri), NS_RDF.type, ???database???))
g.add((DEMARC, NS_DCTERMS.title, Literal("Database of Boundary Disputes in the Early Roman Empire")))
g.add((DEMARC, NS_DCTERMS.creator, TOM))
# more?

# content

CREATOR = NS_DCTERMS["creator"]
NS_DEMARC = Namespace('https://paregorios.org/demarc/')
DINST = NS_DEMARC["DemarcationInstance"]

instance_ids = set()

for instance_head in instance_headings:
    bookmark_start = instance_head.find(".//text:bookmark-start", xml_namespaces)
    if bookmark_start is not None:
        instance_id = bookmark_start.get(f'{{{xml_namespaces["text"]}}}name')
        if instance_id.startswith("INST"):
            num = instance_id[4:]  # Extract the number after "INST"
            if num.isdigit():
                if instance_id not in instance_ids:
                    instance_ids.add(instance_id)
                    demarc_instance_uri = NS_DEMARC[instance_id]
                    g.add((demarc_instance_uri, NS_RDF.type, DINST))
                    g.add((demarc_instance_uri, NS_DCTERMS.isPartOf, DEMARC))
                    raw_label = ''.join(instance_head.itertext())
                    if not raw_label:
                        raise RuntimeError(f"Instance {instance_id} has no rawlabel")
                    label = " ".join([s.strip() for s in (' '.join(''.join(instance_head.itertext()).split())).split(":") if s.strip() != instance_id])
                    if label:
                        #print(label + f"\n{etree.tostring(instance_head, pretty_print=True).decode().replace("<", "\n<")}")
                        # print(label)
                        g.add((demarc_instance_uri, NS_RDFS.label, Literal(label, lang="en")))
                        continue
                    else:
                        raise RuntimeError(f"Instance {instance_id} has no label:\n{etree.tostring(instance_head, pretty_print=True).decode().replace("<", "\n<")}")
    else:
        raise RuntimeError("Abject failure: failed to find bookmark_start")
    raise RuntimeError("Abject failure: undefined")
g.bind("demarc", NS_DEMARC)
print(g.serialize(format="turtle"))


@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix demarc: <https://paregorios.org/demarc/> .
@prefix foaf: <http://xmlns.com/foaf/0.1/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

demarc:INST100 a demarc:DemarcationInstance ;
    rdfs:label "Boundary Dispute Involving Ardea"@en ;
    dcterms:isPartOf demarc:db .

demarc:INST101 a demarc:DemarcationInstance ;
    rdfs:label "Dispute over Site, Ownership and Boundaries between Ostia and Volussius Crocus"@en ;
    dcterms:isPartOf demarc:db .

demarc:INST103 a demarc:DemarcationInstance ;
    rdfs:label "Restoration of the Boundaries of the Fields Consecrated to Diana Tifatina"@en ;
    dcterms:isPartOf demarc:db .

demarc:INST108 a demarc:DemarcationInstance ;
    rdfs:label "Casting a Spell on the Governor in Hispania"@en ;
    dcterms:isPartOf demarc:db .

demarc:INST109 a demarc:DemarcationInstance ;
    rdfs:label "Authoritative Demarcation between the Viennenses and the Ceutrones"@en ;
    dcterms:isPartOf demarc